<a href="https://colab.research.google.com/github/pavnastia/ab-testing-statistical-analysis/blob/main/ab_testing_statistical_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[View A/B Testing Results on Google Drive](https://drive.google.com/file/d/1N0rCcgKf9Pp17WMNOKEGFSHG0zhxHEy9/view?usp=sharing)

[View Interactive Tableau Dashboard](https://public.tableau.com/app/profile/anastasiia.pavlenko/viz/Significance_17915712118370/Dashboard1?publish=yes)

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive, files
from statsmodels.stats.proportion import proportions_ztest
from google.colab import auth
from google.cloud import bigquery

# Authenticate Google account
auth.authenticate_user()

# Create a connection to the BigQuery project
bq = bigquery.Client(project="data-analytics-mate")

# Write SQL query to create the final dataset
query = """
WITH session_info AS (
  SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
  FROM `DA.ab_test` ab
  JOIN `DA.session` s ON ab.ga_session_id = s.ga_session_id
  JOIN `DA.session_params` sp ON sp.ga_session_id = ab.ga_session_id
),

session_with_orders AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `DA.order` o
    JOIN session_info ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),

events AS (
  SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      sp.event_name,
      COUNT(sp.ga_session_id) AS event_cnt
  FROM `DA.event_params` sp
  JOIN session_info ON sp.ga_session_id = session_info.ga_session_id
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      sp.event_name
),

session AS (
  SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(session_info.ga_session_id) AS session_cnt
  FROM session_info
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
 ),

account AS (
  SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
  FROM `DA.account_session` acs
  JOIN session_info ON acs.ga_session_id = session_info.ga_session_id
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
)

SELECT
      session_with_orders.date,
      session_with_orders.country,
      session_with_orders.device,
      session_with_orders.continent,
      session_with_orders.channel,
      session_with_orders.test,
      session_with_orders.test_group,
      'session with orders' AS event_name,
      session_with_orders.session_with_orders AS value
FROM session_with_orders

UNION ALL

SELECT
      events.date,
      events.country,
      events.device,
      events.continent,
      events.channel,
      events.test,
      events.test_group,
      event_name,
      event_cnt AS value
FROM events

UNION ALL

SELECT
      session.date,
      session.country,
      session.device,
      session.continent,
      session.channel,
      session.test,
      session.test_group,
      'session' AS event_name,
      session_cnt AS value
FROM session

UNION ALL

SELECT
      account.date,
      account.country,
      account.device,
      account.continent,
      account.channel,
      account.test,
      account.test_group,
      'new account' AS event_name,
      new_account_cnt AS value
FROM account

"""

# Execute the query and load the result into a Pandas DataFrame
df = bq.query(query).to_dataframe(
    create_bqstorage_client=False
)


# Prepare data
df.columns = df.columns.str.strip()
df['event_name'] = df['event_name'].str.strip().str.lower()

# Define metrics
metrics = {
    'add_payment_info': 'add_payment_info',
    'add_shipping_info': 'add_shipping_info',
    'begin_checkout': 'begin_checkout',
    'new_accounts': 'new account'
}

# Aggregate data across the entire test period
summary = df.groupby(
    ['test', 'test_group', 'event_name']
)['value'].sum()

results = []

# Calculate statistical significance for each test and metric
for test_number in sorted(df['test'].unique()):
    for metric, event in metrics.items():

        # Get event and session counts for both groups
        a = summary.get((test_number, 1, event), 0)
        b = summary.get((test_number, 2, event), 0)
        n_a = summary.get((test_number, 1, 'session'), 0)
        n_b = summary.get((test_number, 2, 'session'), 0)

        # Validate the counts
        if min(n_a, n_b) <= 0 or a > n_a or b > n_b:
            raise ValueError(f'Invalid counts in test {test_number}: {metric}')

        # Calculate conversion rates and relative change
        cr_a, cr_b = a / n_a, b / n_b
        change = (cr_b / cr_a - 1) * 100 if cr_a > 0 else np.nan

        # Perform a two-sided Z-test
        if a + b in (0, n_a + n_b):
            z_stat, p_value = 0.0, 1.0
        else:
            z_stat, p_value = proportions_ztest(
                [b, a], [n_b, n_a], alternative='two-sided'
            )

        results.append({
            'test_number': test_number,
            'metric': f'{metric} / session',
            'numerator_event': event,
            'denominator_event': 'session',
            'numerator_count_test': b,
            'denominator_count_test': n_b,
            'conversion_rate_test': cr_b,
            'numerator_count_control': a,
            'denominator_count_control': n_a,
            'conversion_rate_control': cr_a,
            'metric_change_%': change,
            'z_stat': z_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        })

# Create and display the final table
df_results = pd.DataFrame(results)
display(df_results)

# Save and download the results
df_results.to_csv('results.csv', index=False, encoding='utf-8-sig')
files.download('results.csv')

,test_number,metric,numerator_event,denominator_event,numerator_count_test,denominator_count_test,conversion_rate_test,numerator_count_control,denominator_count_control,conversion_rate_control,metric_change_%,z_stat,p_value,significant
0,1,add_payment_info / session,add_payment_info,session,2229,45193,0.049322,1988,45362,0.043825,12.542021,3.924884,0.000087,True
1,1,add_shipping_info / session,add_shipping_info,session,3221,45193,0.071272,3034,45362,0.066884,6.560481,2.603571,0.009226,True
2,1,begin_checkout / session,begin_checkout,session,4021,45193,0.088974,3784,45362,0.083418,6.660587,2.978783,0.002894,True
3,1,new_accounts / session,new account,session,3681,45193,0.081451,3823,45362,0.084278,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info / session,add_payment_info,session,2409,50244,0.047946,2344,50637,0.046290,3.576911,1.240994,0.214608,False
5,2,add_shipping_info / session,add_shipping_info,session,3510,50244,0.069859,3480,50637,0.068724,1.650995,0.709557,0.477979,False
6,2,begin_checkout / session,begin_checkout,session,4313,50244,0.085841,4262,50637,0.084168,1.988164,0.952898,0.340642,False
7,2,new_accounts / session,new account,session,4184,50244,0.083274,4165,50637,0.082252,1.241934,0.588793,0.556000,False
8,3,add_payment_info / session,add_payment_info,session,3697,70439,0.052485,3623,70047,0.051722,1.474630,0.643172,0.520112,False
9,3,add_shipping_info / session,add_shipping_info,session,5188,70439,0.073652,5298,70047,0.075635,-2.621211,-1.413727,0.157442,False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>